In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [5]:
!pip install -q timm scikit-learn matplotlib seaborn pandas pillow tqdm

In [7]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [8]:
import os

DATA_DIR = "/kaggle/input"

for root, dirs, files in os.walk(DATA_DIR):
    print(root)
    print("Folders:", dirs[:5])
    print("Files:", files[:5])
    print("-" * 60)

/kaggle/input
Folders: ['datasets']
Files: []
------------------------------------------------------------
/kaggle/input/datasets
Folders: ['div456']
Files: []
------------------------------------------------------------
/kaggle/input/datasets/div456
Folders: ['indian-saree-patterns']
Files: []
------------------------------------------------------------
/kaggle/input/datasets/div456/indian-saree-patterns
Folders: ['valid', 'test', 'train']
Files: ['README.dataset.txt', 'README.roboflow.txt']
------------------------------------------------------------
/kaggle/input/datasets/div456/indian-saree-patterns/valid
Folders: ['Banarasi', 'Pichwai', 'Bandhani', 'Ikat']
Files: []
------------------------------------------------------------
/kaggle/input/datasets/div456/indian-saree-patterns/valid/Banarasi
Folders: []
Files: ['images7_jpg.rf.548124a6f72034f6d9d68a5ee7a0ec38.jpg', 'X2A4519_321be3c6-86a5-40a0-b208-6183f2290c91_jpg.rf.a5fe3966dcd7917a759342797b9ca3a8.jpg', 'images138_jpg.rf.0532a37

In [9]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

image_paths = []

for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        if file.lower().endswith(IMAGE_EXTENSIONS):
            image_paths.append(os.path.join(root, file))

print("Total images:", len(image_paths))

for path in image_paths[:20]:
    print(path)

Total images: 1468
/kaggle/input/datasets/div456/indian-saree-patterns/valid/Banarasi/images7_jpg.rf.548124a6f72034f6d9d68a5ee7a0ec38.jpg
/kaggle/input/datasets/div456/indian-saree-patterns/valid/Banarasi/X2A4519_321be3c6-86a5-40a0-b208-6183f2290c91_jpg.rf.a5fe3966dcd7917a759342797b9ca3a8.jpg
/kaggle/input/datasets/div456/indian-saree-patterns/valid/Banarasi/images138_jpg.rf.0532a3771bb1b61fa0b473388efbe1cc.jpg
/kaggle/input/datasets/div456/indian-saree-patterns/valid/Banarasi/images168_jpg.rf.d64adabba8f41b5f3ab9878d33be6474.jpg
/kaggle/input/datasets/div456/indian-saree-patterns/valid/Banarasi/images172_jpg.rf.df69457214f062fabc3abc88494e8725.jpg
/kaggle/input/datasets/div456/indian-saree-patterns/valid/Banarasi/IF1B1HM123050602_Pink_Banarasi_Silk_Fabric_jpg.rf.cfb3c7cd97909675586b9321f0e135d3.jpg
/kaggle/input/datasets/div456/indian-saree-patterns/valid/Banarasi/images142_jpg.rf.bbbec6ce70a0941c05f7b553dfc851f3.jpg
/kaggle/input/datasets/div456/indian-saree-patterns/valid/Banarasi/i

In [10]:
records = []

for path in image_paths:

    parent_folder = os.path.basename(os.path.dirname(path))

    records.append({
        "image_path": path,
        "design_id": parent_folder
    })

df = pd.DataFrame(records)

print(df.head())
print("Number of images:", len(df))
print("Number of designs:", df["design_id"].nunique())

                                          image_path design_id
0  /kaggle/input/datasets/div456/indian-saree-pat...  Banarasi
1  /kaggle/input/datasets/div456/indian-saree-pat...  Banarasi
2  /kaggle/input/datasets/div456/indian-saree-pat...  Banarasi
3  /kaggle/input/datasets/div456/indian-saree-pat...  Banarasi
4  /kaggle/input/datasets/div456/indian-saree-pat...  Banarasi
Number of images: 1468
Number of designs: 4


In [11]:
print(df["design_id"].value_counts().head(20))

design_id
Banarasi    489
Ikat        342
Pichwai     321
Bandhani    316
Name: count, dtype: int64


In [12]:
design_mapping = {
    "red_saree_01.jpg": "design001",
    "blue_saree_01.jpg": "design001",
    "green_saree_01.jpg": "design001",

    "red_saree_02.jpg": "design002"
}

In [13]:
design_names = sorted(df["design_id"].unique())

design_to_label = {
    design: idx
    for idx, design in enumerate(design_names)
}

df["label"] = df["design_id"].map(design_to_label)

print(df.head())

                                          image_path design_id  label
0  /kaggle/input/datasets/div456/indian-saree-pat...  Banarasi      0
1  /kaggle/input/datasets/div456/indian-saree-pat...  Banarasi      0
2  /kaggle/input/datasets/div456/indian-saree-pat...  Banarasi      0
3  /kaggle/input/datasets/div456/indian-saree-pat...  Banarasi      0
4  /kaggle/input/datasets/div456/indian-saree-pat...  Banarasi      0


In [14]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

gallery_df, query_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Train:", len(train_df))
print("Gallery:", len(gallery_df))
print("Query:", len(query_df))

Train: 1027
Gallery: 220
Query: 221


In [15]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0)
    ),
    transforms.ColorJitter(
        brightness=0.4,
        contrast=0.4,
        saturation=0.8,
        hue=0.3
    ),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [16]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [17]:
class SareeDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_path = row["image_path"]
        label = int(row["label"])

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [18]:
train_dataset = SareeDataset(
    train_df,
    train_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

In [19]:
import torchvision.models as models

class SareeEmbeddingModel(nn.Module):

    def __init__(self, embedding_dim=128):

        super().__init__()

        self.backbone = models.resnet18(
            weights=models.ResNet18_Weights.DEFAULT
        )

        input_features = self.backbone.fc.in_features

        self.backbone.fc = nn.Identity()

        self.embedding = nn.Sequential(
            nn.Linear(input_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, embedding_dim)
        )

    def forward(self, x):

        features = self.backbone(x)

        embedding = self.embedding(features)

        embedding = F.normalize(
            embedding,
            p=2,
            dim=1
        )

        return embedding

In [20]:
criterion = nn.TripletMarginLoss(
    margin=0.3,
    p=2
)

In [22]:
class TripletSareeDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

        self.labels = self.df["label"].values

        self.label_to_indices = {}

        for label in np.unique(self.labels):

            self.label_to_indices[label] = np.where(
                self.labels == label
            )[0]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):

        anchor_row = self.df.iloc[index]

        anchor_label = anchor_row["label"]

        positive_candidates = self.label_to_indices[
            anchor_label
        ]

        positive_index = index

        while positive_index == index:
            positive_index = np.random.choice(
                positive_candidates
            )

        negative_label = np.random.choice(
            [
                x for x in self.label_to_indices.keys()
                if x != anchor_label
            ]
        )

        negative_index = np.random.choice(
            self.label_to_indices[negative_label]
        )

        anchor = Image.open(
            self.df.iloc[index]["image_path"]
        ).convert("RGB")

        positive = Image.open(
            self.df.iloc[positive_index]["image_path"]
        ).convert("RGB")

        negative = Image.open(
            self.df.iloc[negative_index]["image_path"]
        ).convert("RGB")

        if self.transform:

            anchor = self.transform(anchor)
            positive = self.transform(positive)
            negative = self.transform(negative)

        return anchor, positive, negative

In [23]:
triplet_dataset = TripletSareeDataset(
    train_df,
    train_transform
)

triplet_loader = DataLoader(
    triplet_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

In [24]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = SareeEmbeddingModel(
    embedding_dim=128
).to(device)

print(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 210MB/s]


cuda


In [26]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [27]:
EPOCHS = 10

train_losses = []

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0.0

    progress = tqdm(
        triplet_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for anchor, positive, negative in progress:

        anchor = anchor.to(device)
        positive = positive.to(device)
        negative = negative.to(device)

        optimizer.zero_grad()

        anchor_embedding = model(anchor)
        positive_embedding = model(positive)
        negative_embedding = model(negative)

        loss = criterion(
            anchor_embedding,
            positive_embedding,
            negative_embedding
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        progress.set_postfix(
            loss=loss.item()
        )

    epoch_loss = running_loss / len(
        triplet_loader
    )

    train_losses.append(epoch_loss)

    print(
        f"Epoch {epoch+1}: "
        f"Loss = {epoch_loss:.4f}"
    )

Epoch 1/10:   0%|          | 0/33 [00:00<?, ?it/s]

Epoch 1: Loss = 0.2016


Epoch 2/10:   0%|          | 0/33 [00:00<?, ?it/s]

Epoch 2: Loss = 0.0839


Epoch 3/10:   0%|          | 0/33 [00:00<?, ?it/s]

...

Epoch 9/10:   0%|          | 0/33 [00:00<?, ?it/s]

Epoch 9: Loss = 0.0122


Epoch 10/10:   0%|          | 0/33 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e6a839f7880>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e6a839f7880>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Epoch 10: Loss = 0.0135


In [28]:
plt.savefig(
    "/kaggle/working/training_loss.png",
    dpi=300,
    bbox_inches="tight"
)

<Figure size 640x480 with 0 Axes>

In [29]:
torch.save(
    model.state_dict(),
    "/kaggle/working/saree_model.pth"
)

In [32]:
gallery_dataset = SareeDataset(
    gallery_df,
    test_transform
)

gallery_loader = DataLoader(
    gallery_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

In [33]:
def generate_embeddings(
    model,
    loader,
    device
):

    model.eval()

    embeddings = []
    labels = []

    with torch.no_grad():

        for images, batch_labels in tqdm(loader):

            images = images.to(device)

            batch_embeddings = model(images)

            embeddings.append(
                batch_embeddings.cpu()
            )

            labels.append(
                batch_labels
            )

    embeddings = torch.cat(
        embeddings
    )

    labels = torch.cat(
        labels
    )

    return embeddings, labels

In [34]:
gallery_embeddings, gallery_labels = generate_embeddings(
    model,
    gallery_loader,
    device
)

print(gallery_embeddings.shape)

  0%|          | 0/7 [00:00<?, ?it/s]

torch.Size([220, 128])


In [35]:
query_dataset = SareeDataset(
    query_df,
    test_transform
)

query_loader = DataLoader(
    query_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

query_embeddings, query_labels = generate_embeddings(
    model,
    query_loader,
    device
)

  0%|          | 0/7 [00:00<?, ?it/s]

In [36]:
similarity_matrix = torch.mm(
    query_embeddings,
    gallery_embeddings.T
)

In [37]:
top1_indices = similarity_matrix.argmax(
    dim=1
)

predicted_labels = gallery_labels[
    top1_indices
]

identification_accuracy = (
    predicted_labels == query_labels
).float().mean().item()

print(
    "Top-1 Identification Accuracy:",
    identification_accuracy
)

Top-1 Identification Accuracy: 0.9909502267837524


In [39]:
top5_indices = similarity_matrix.topk(
    5,
    dim=1
).indices

correct = 0

for i in range(len(query_labels)):

    retrieved_labels = gallery_labels[
        top5_indices[i]
    ]

    if query_labels[i] in retrieved_labels:
        correct += 1

top5_accuracy = correct / len(query_labels)

print(
    "Top-5 Accuracy:",
    top5_accuracy
)

Top-5 Accuracy: 0.9909502262443439


In [40]:
def cosine_similarity(
    embedding1,
    embedding2
):

    return F.cosine_similarity(
        embedding1.unsqueeze(0),
        embedding2.unsqueeze(0)
    ).item()

In [41]:
positive_pairs = []
negative_pairs = []

for i in range(len(query_labels)):

    for j in range(len(gallery_labels)):

        similarity = similarity_matrix[i, j].item()

        same_design = (
            query_labels[i].item()
            ==
            gallery_labels[j].item()
        )

        if same_design:
            positive_pairs.append(similarity)

        else:
            negative_pairs.append(similarity)

In [42]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    true_labels,
    predicted_labels
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Verification Confusion Matrix")

plt.show()

NameError: name 'true_labels' is not defined

In [44]:
def cosine_similarity(
    embedding1,
    embedding2
):

    return F.cosine_similarity(
        embedding1.unsqueeze(0),
        embedding2.unsqueeze(0)
    ).item()

In [45]:
positive_pairs = []
negative_pairs = []

for i in range(len(query_labels)):

    for j in range(len(gallery_labels)):

        similarity = similarity_matrix[i, j].item()

        same_design = (
            query_labels[i].item()
            ==
            gallery_labels[j].item()
        )

        if same_design:
            positive_pairs.append(similarity)

        else:
            negative_pairs.append(similarity)

In [46]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    true_labels,
    predicted_labels
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Verification Confusion Matrix")

plt.show()

ValueError: zero-size array to reduction operation fmin which has no identity

<Figure size 600x500 with 0 Axes>